# Fake multimodal

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
exp = "simulated_multimodal"
results_path = f"../results/{exp}"

# load results
results_df = pd.read_csv(f"{results_path}/{exp}_results.csv")

# in the df, replace "_" with " " 
results_df['split'] = results_df['split'].str.replace('_', ' ')
results_df.columns = results_df.columns.str.replace('_', ' ')
results_df['model'] = results_df['model'].str.replace('RFMALI_WIP', 'FoSTA')


# # convert relevant columns to numeric
num_cols = results_df.columns.difference(['model', 'dataset', 'split', 'seed'])
# results_df[num_cols] = results_df[num_cols].apply(pd.to_numeric, errors='coerce')


# take the abs for silhouette domain because it should be low, doesnt matter the sign
results_df['Silhouette domain'] = results_df['Silhouette domain'].abs()

results_df

average over all datasets

In [ ]:
summary_df = results_df.groupby(['split','model'])[num_cols].mean()
summary_df_std = results_df.groupby(['split','model'])[num_cols].std()
summary_df

In [ ]:
summary_df_std

# print it to a table

subset to the methods/metrics we want

In [ ]:
# clean summary_df 

def clean_summary_df(df):
    # remove "RFMALI" 
    df = df.loc[df.index.get_level_values('model') != 'RFMALI']

    cols_to_keep = ["Accuracy missing", "Alignment score", "FOSCTTM"]
    # rename Accuracy missing to Accuracy
    df = df[cols_to_keep]
    df = df.rename(columns={"Accuracy missing": "Accuracy"})
    return df

summary_df = clean_summary_df(summary_df)
summary_df_std = clean_summary_df(summary_df_std)

In [ ]:
# Function to format the top 3 values: bold, underline, italic (adjusted for lower is better columns)
def format_top_3(df):
    formatted_df = df.copy()
    
    # List of columns where lower values are better
    lower_is_better_columns = ['FOSCTTM', 'Silhouette domain']
    
    # Group by 'split' to format the top 3 values in each split
    for split, group in df.groupby('split'):
        for column in group.columns:  # Start from the first numerical column
            if column in lower_is_better_columns:
                # Find the smallest 3 values (lower is better)
                top_3 = group[column].nsmallest(3).values
            else:
                # Find the largest 3 values (higher is better)
                top_3 = group[column].nlargest(3).values
            
            if len(top_3) >= 3:
                # Apply formatting: bold for best, underline for second, italic for third
                formatted_df.loc[group.index, column] = group[column].apply(
                    lambda x: f"\\textbf{{{x:.3f}}}" if x == top_3[0] else 
                              (f"\\underline{{{x:.3f}}}" if x == top_3[1] else 
                               (f"\\textit{{{x:.3f}}}" if x == top_3[2] else f"{x:.3f}"))
                )
    
    return formatted_df

In [ ]:
summary_df_fmt = format_top_3(summary_df)


In [ ]:
# add std

# for col in summary_df.columns:
#     summary_df_fmt[col] = (
#         summary_df[col].round(3).astype(str)
#         + r" {\small $\pm$ "
#         + summary_df_std[col].round(3).astype(str)
#         + "}"
#     )

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(summary_df_fmt.to_latex(
        escape=False,
        multirow=True,
        float_format="%.3f"
    )) 

# then copy the output latex table into a .tex file for inclusion in the paper

# bar plots

In [ ]:
for col in num_cols:
    plt.figure(figsize=(10,6))
    sns.barplot(data=results_df, x='model', y=col, hue='split')
    
    if col == "Silhouette domain":
        plt.figure(figsize=(10,6))
        sns.barplot(data=results_df[(results_df['model'] != "KEMArbf") & (results_df['model'] != "KEMAlin")], x='model', y=col, hue='split')
        plt.show()

# grouped by split

In [ ]:
for col in num_cols:
    plt.figure(figsize=(10,6))
    sns.barplot(data=results_df, x='split', y=col, hue='model')
    
    if col == "Silhouette domain":
        plt.figure(figsize=(10,6))
        sns.barplot(data=results_df[(results_df['model'] != "KEMArbf") & (results_df['model'] != "KEMAlin")], x='split', y=col, hue='model')
        plt.show()

# checking the failed runs

In [ ]:
nas = results_df[results_df['Silhouette domain'].isna()][['split', 'dataset', 'model']]
nas

In [ ]:
print(nas['split'].value_counts())
print(nas['dataset'].value_counts())
print(nas['model'].value_counts())

In [ ]:
datasets_path = "../data"
data_name = "breast_cancer"
df = pd.read_csv(f"{datasets_path}/{data_name}.csv")

In [ ]:
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents) if (p/"src").is_dir())))
from utils.utils import dataprep
df, labels = dataprep(df) 

In [ ]:
df

In [ ]:
df.isna().sum()